# Reconstructed EMG CNN

Trains `TDSConvCTC` on **AE-decoded (reconstructed) EMG signals** from `*_recons_v3.hdf5`.
Unlike the latent CNN, this operates in the decoded signal space — closer to raw EMG
but at 32× lower temporal resolution (62.5 Hz instead of 2000 Hz).

**Data:** `data/*_recons_v3.hdf5` — AE-reconstructed EMG @ 62.5 Hz, shape `(T, 16)` per hand  
**Model:** `Flatten(2×16→32) → Linear(32→num_features) → ReLU → TDSConvEncoder → CTC`  
**Checkpoint:** `Playground_Ben/checkpoints/best_recons_cnn.pt`  
**Results:** Val CER **83.33%** · Test CER **81.96%** (150 epochs)

```bash
source /home/benforbes/emg2qwerty/venv/bin/activate
cd ~/C247_mumbikaijonathanben && jupyter notebook
```

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path
import torch

REPO_ROOT = Path(os.getcwd())
while not (REPO_ROOT / 'emg2qwerty').is_dir():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

PLAYGROUND = REPO_ROOT / 'Playground_Ben'
SCRIPT     = PLAYGROUND / 'scripts/train_recons_cnn.py'
CHECKPOINT = PLAYGROUND / 'checkpoints/best_recons_cnn.pt'

# Verify reconstructed data files exist
recons_files = list((REPO_ROOT / 'data').glob('*_recons_v3.hdf5'))
print(f'Repo root: {REPO_ROOT}')
print(f'Recons data files found: {len(recons_files)}')
for f in recons_files:
    print(f'  {f.name}')

## 2. Data Loader Overview

`Playground_Ben/recons_data_utils.py` provides `get_recons_dataloaders()`, which mirrors
the API of `Playground_Kai/data_utils.py` and reads the reconstructed HDF5 schema:
- `emg_left`: `(T, 16)` float32
- `emg_right`: `(T, 16)` float32

Batches: `{'inputs': (T, N, 2, 16), 'targets': ..., 'input_lengths': ..., 'target_lengths': ...}`

In [ ]:
# Preview data loader stats
from Playground_Ben.recons_data_utils import get_recons_dataloaders

loaders = get_recons_dataloaders(
    data_root=REPO_ROOT / 'data',
    config_path=REPO_ROOT / 'config/user/single_user.yaml',
    window_length=250,
    stride=250,
    batch_size=32,
    num_workers=0,
)
print(f"Train windows : {len(loaders['train'].dataset):,}")
print(f"Val   windows : {len(loaders['val'].dataset):,}")
print(f"Test  windows : {len(loaders['test'].dataset):,}")

batch = next(iter(loaders['train']))
print(f"\nBatch inputs shape : {batch['inputs'].shape}   (T, N, bands=2, channels=16)")

## 3. Train

In [ ]:
result = subprocess.run(
    [sys.executable, str(SCRIPT),
     '--epochs', '150',
     '--lr', '1e-3',
     '--batch-size', '32',
     '--window-length', '250',
     '--stride', '250'],
    cwd=str(REPO_ROOT)
)
print(f'Exit code: {result.returncode}')

## 4. Inspect Checkpoint

In [ ]:
if CHECKPOINT.exists():
    ckpt = torch.load(CHECKPOINT, map_location='cpu')
    print('Reconstructed EMG CNN — Best Checkpoint')
    print(f'  Epoch   : {ckpt["epoch"]}')
    print(f'  Val CER : {ckpt["val_cer"]:.2f}%')
    print(f'  File    : {CHECKPOINT}')
else:
    print('Checkpoint not found — run the training cell first.')

## 5. Architecture Summary

```
Input:  (T, N, 2, 16)  — reconstructed EMG bands at 62.5 Hz
  └─ Flatten(start_dim=2)         → (T, N, 32)
  └─ Linear(32 → num_features)   +  ReLU
  └─ TDSConvEncoder(num_features, block_channels, kernel_width)
  └─ Linear(num_features → num_classes)  +  LogSoftmax
Output: (T', N, 71)   — CTC log-probs
```

**Comparison with latent AE CNN:**
- Latent CNN input: 1024-dim compressed vectors (information bottleneck)
- Recons CNN input: 32-dim decoded signals (post-bottleneck, signal-space)
- Both operate at 62.5 Hz — the AE encoder frame rate